# 01 - Exploratory Data Analysis
Run `python data/download_data.py` first. Works from the repo root or from `notebooks/` (also in Colab after cloning the repo).

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.data import load_config, load_raw
cfg = load_config()
raw = pd.read_csv(cfg["data"]["raw_path"])
df = load_raw(cfg["data"]["raw_path"])
df.head()

## 1. Size, types, missing values, duplicates

In [ ]:
print("Raw shape:", raw.shape, "| after cleaning:", df.shape)
print("Date range:", df['date'].min(), "->", df['date'].max())
print("Missing values per column:"); print(raw.isna().sum())
print("Duplicated rows (raw):", int(raw.duplicated().sum()))
print("Duplicated timestamps (raw):", int(raw['date'].duplicated().sum()))
print("Time step between readings:"); print(df['date'].diff().dropna().value_counts())
raw.dtypes

**Conclusion (fill from the output above):** The dataset has ___ rows and ___ columns, covering ___ to ___ at 15-minute steps. Missing values: ___. Duplicated rows/timestamps: ___. Therefore the cleaning decision is: ___.

## 2. Target distribution and outliers

In [ ]:
s = df['Usage_kWh']
print(s.describe())
q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
out = ((s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)).sum()
print(f"IQR outliers: {out} ({out/len(s):.2%}) | zero readings: {(s==0).sum()}")
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].hist(s, bins=60); ax[0].set_title('Usage_kWh distribution'); ax[0].set_xlabel('kWh')
ax[1].boxplot(s, vert=False); ax[1].set_title('Boxplot')
plt.show()

**Conclusion:** Usage_kWh ranges from ___ to ___ kWh (mean ___, median ___). The distribution is ___ (e.g. right-skewed, or two peaks for idle and production). Outliers by IQR: ___ %. Decision: ___ (e.g. keep them, they are real production peaks; MAE is less sensitive to them than RMSE).

## 3. Feature ranges and categorical columns

In [ ]:
print(df.describe().T[['min','mean','max','std']].round(2))
for c in ['WeekStatus', 'Day_of_week', 'Load_Type']:
    if c in df:
        print(f"\n{c}:"); print(df[c].value_counts())

**Conclusion:** The numeric features have very different ranges (___ to ___), so they must be scaled (standard scaler, fitted on train only). Text columns: WeekStatus, Day_of_week, Load_Type, so the dataset has mixed feature types; we encode the time information as numeric time features instead.

In [ ]:
if 'Load_Type' in df:
    df.boxplot(column='Usage_kWh', by='Load_Type', figsize=(6,4)); plt.suptitle(''); plt.title('Usage_kWh by Load_Type'); plt.show()

**Conclusion:** Usage_kWh is ___ for Light_Load, ___ for Medium_Load and ___ for Maximum_Load, so Load_Type is strongly related to consumption. It is derived from consumption, so it is NOT used as an input.

## 4. Correlations

In [ ]:
num = df.select_dtypes('number').drop(columns=['NSM'], errors='ignore')
corr = num.corr()
fig, ax = plt.subplots(figsize=(7,6))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=7)
fig.colorbar(im); plt.show()
print(corr['Usage_kWh'].sort_values(ascending=False).round(3))

**Conclusion:** Usage_kWh is most correlated with ___ (r = ___) and ___ (r = ___). CO2 follows consumption closely, which is why carbon analysis can be built on the forecast. Weak features: ___.

## 5. Time patterns

In [ ]:
fig, ax = plt.subplots(figsize=(12,3))
ax.plot(df['date'], df['Usage_kWh'], lw=0.3); ax.set_title('Usage_kWh over the year'); ax.set_ylabel('kWh')
plt.show()

**Conclusion:** Over the year consumption ___ (stable / seasonal / drops in ___). Visible gaps or unusual periods: ___.

In [ ]:
d = df.assign(hour=df['date'].dt.hour, weekend=df['date'].dt.dayofweek>=5)
prof = d.groupby(['weekend','hour'])['Usage_kWh'].mean().unstack(0)
prof.columns = ['Weekday','Weekend']
prof.plot(figsize=(8,4), title='Average daily profile (kWh per 15 min)'); plt.xlabel('Hour'); plt.show()
prof.round(1).T

**Conclusion:** On weekdays consumption rises at ___ h and falls at ___ h, with a peak around ___ kWh. On weekends it is ___ (lower / flat). Decision: add hour, day-of-week and weekend features.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
for ax, start in zip(axes, ['2018-01-08', '2018-05-07', '2018-10-08']):
    w = df[(df['date'] >= start) & (df['date'] < pd.Timestamp(start) + pd.Timedelta(days=7))]
    ax.plot(w['date'], w['Usage_kWh']); ax.set_title(f'One week starting {start}')
plt.tight_layout(); plt.show()

**Conclusion:** The weekly pattern is ___ (repeats every week, weekends lower). Differences between the three weeks: ___.

In [ ]:
s = df['Usage_kWh']
ac = [s.autocorr(l) for l in range(1, 7*96+1)]
fig, ax = plt.subplots(figsize=(10,3)); ax.plot(range(1, len(ac)+1), ac)
for k in range(1,8): ax.axvline(96*k, color='r', ls=':', lw=0.7)
ax.set_title('Autocorrelation (red lines = multiples of 24 h)'); ax.set_xlabel('lag (15-min steps)'); plt.show()
print('autocorr at lag 96 (24 h):', round(s.autocorr(96),3), '| lag 672 (7 days):', round(s.autocorr(672),3))

**Conclusion:** Autocorrelation at lag 96 (24 h) = ___ and at lag 672 (7 days) = ___. Strong daily repetition, so a 24 h input window (96 steps) is justified and 'tomorrow = today' is the right trivial baseline.

## 6. Do train / validation / test look alike?

In [ ]:
n = len(df); i1, i2 = int(n*.70), int(n*.85)
for name, part in [('train', df.iloc[:i1]), ('val', df.iloc[i1:i2]), ('test', df.iloc[i2:])]:
    print(f"{name:5s} {part['date'].min().date()} -> {part['date'].max().date()}  mean={part['Usage_kWh'].mean():6.2f}  std={part['Usage_kWh'].std():6.2f}  weekend share={(part['date'].dt.dayofweek>=5).mean():.2f}")

**Conclusion:** Train / validation / test have mean ___ / ___ / ___ and weekend share ___ / ___ / ___. The periods are ___ (similar / different), so a distribution shift is ___ expected; this is discussed in the limitations.

## Findings table (fill from the outputs above, then delete this line)

| # | Finding (with the number you saw) | Modelling decision it leads to |
|---|---|---|
| 1 | Missing values / duplicates: ... | e.g. no imputation needed / drop duplicates |
| 2 | Target is skewed / has outliers: ... | e.g. keep outliers (real peaks), use MAE |
| 3 | Daily cycle: autocorrelation at lag 96 = ... | 24 h input window (96 steps); seasonal-naive baseline |
| 4 | Weekday vs weekend profile differs: ... | add `is_weekend` and day-of-week features |
| 5 | Features most correlated with Usage_kWh: ... | keep them as inputs (only past values are used) |
| 6 | Train / val / test statistics differ: ... | report val and test separately; discuss distribution shift |